# Build `notebooks/data/` from raw 1000 Genomes sources

Run this notebook **once** to (re)build `notebooks/data/`, the small, self-contained data bundle
`simplified_notebook.ipynb` reads from. It never touches the pre-built `1kG_high_coverage` dataset
or anything under `genomics.*` -- everything is derived straight from the same raw 1000 Genomes
sources `genomics.workflows.dataset_builders.non_longevous.build_non_longevous_dataset` and
`build_window_and_predict` use:

- **Pedigree/population metadata**: `docs/historical/1000-genomes/1000_genomes_metadata.csv`
  (`FamilyID,SampleID,FatherID,MotherID,Sex,Population,Superpopulation`; checked into this repo).
- **Gene coordinates**: the public GENCODE v46 GTF (same URL `simplified_notebook.ipynb` already uses).
- **Reference genome + genotypes**: the GRCh38 FASTA and per-chromosome 1000G high-coverage phased
  VCFs, streamed directly from the **public 1000 Genomes FTP** by default (plain HTTP, byte-range
  requests) -- `bcftools`/`samtools` fetch only the small regions each gene's window needs (~5s per
  512kb window), not the whole multi-GB files. **No local genomics data is required**: clone this
  repo, install dependencies, run this notebook, and `notebooks/data/` is ready.

If this machine happens to already have the pre-built `1kG_high_coverage` dataset (checked
automatically in Step 2), that's used instead where it's strictly better -- bit-exact window
boundaries and, in Step 5, its already-computed AlphaGenome predictions (avoiding ~23,600 API
calls). That's an optional local speedup, never a requirement.

Only *this* notebook needs `bcftools`/`samtools` on PATH and network access (or local copies of the
reference FASTA/VCFs). `simplified_notebook.ipynb` itself only ever reads `notebooks/data/` afterward.

Writes, under `notebooks/data/`:
- `experiment.json` -- genes, ontology terms, pigmentation class_map, window size
- `genes.json` -- `{gene: {chrom, start, end}}`, each gene's GTF `"gene"` feature resized to the
  window size (the same algorithm `build_window_and_predict.py` used to build the original dataset)
- `individuals.json` -- `{sample_id: {population, superpopulation, sex, family_id}}`, restricted to
  individuals whose population is in `class_map`
- `references/<gene>/ref.window.fa`
- `variants/<gene>.vcf.gz(+.tbi)` -- sliced to that gene's window, restricted to those individuals

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("/home/breno/I2CA/genomics")
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from utils import prepare_data as pd_setup

print(f"Metadata CSV : {pd_setup.DEFAULT_METADATA_CSV}")
print(f"Reference FASTA: {pd_setup.DEFAULT_REF_FASTA}")
print(f"VCF pattern    : {pd_setup.DEFAULT_VCF_PATTERN}")
print(f"Genes          : {pd_setup.GENES}")
print(f"Class map      : {pd_setup.CLASS_MAP}")

pd_setup.require_tools()
print("bcftools/samtools found on PATH.")

## Step 1: metadata

Load the 1000 Genomes pedigree CSV and summarize the individuals belonging to the pigmentation
`class_map` populations (mirrors `build_non_longevous_dataset.py`'s own metadata report).

In [ ]:
metadata_df = pd_setup.load_metadata()
print(f"Loaded {len(metadata_df)} individuals from {pd_setup.DEFAULT_METADATA_CSV.name}")

summary_df = pd_setup.summarize_class_populations(metadata_df, pd_setup.CLASS_MAP)
summary_df

## Step 2: gene windows

Two ways to resolve each gene's window (chrom/start/end), auto-selected below by whether
`DEFAULT_PRECOMPUTED_DATASET_DIR` exists on this machine:

- **From the pre-built dataset** (used automatically if present): copies its own
  `window_metadata.json` / `ref.window.fa` directly. Required for exact alignment if you'll also
  import its precomputed predictions in Step 5.
- **From the GTF** (used otherwise -- e.g. on a fresh clone with no local genomics data): the gene's
  `"gene"` feature (whole-gene span across every isoform), resized (centered) to the fixed window
  size -- the same algorithm `build_window_and_predict.py` used to build the windows the original
  `1kG_high_coverage` dataset shipped with. A `resize()` here can land 1-2bp off from what that
  dataset actually used (rounding) -- inconsequential on its own, but why the pre-built dataset's
  own numbers are preferred when available.

In [ ]:
# Auto-detect: use this machine's pre-built dataset if present, else derive windows from the GTF
# (streaming the reference FASTA from the public 1000G FTP -- see markdown above). Force one or
# the other by setting PRECOMPUTED_DATASET_DIR directly instead of relying on autodetection.
PRECOMPUTED_DATASET_DIR = pd_setup.DEFAULT_PRECOMPUTED_DATASET_DIR
if not PRECOMPUTED_DATASET_DIR.exists():
    print(f"{PRECOMPUTED_DATASET_DIR} not found on this machine -- deriving windows from the GTF "
          "and streaming the reference FASTA from the public 1000G FTP instead.")
    PRECOMPUTED_DATASET_DIR = None

gene_windows = {}
if PRECOMPUTED_DATASET_DIR is not None:
    print(f"Sourcing gene windows from: {PRECOMPUTED_DATASET_DIR}")
    for gene in pd_setup.GENES:
        chrom, start, end = pd_setup.load_dataset_window(PRECOMPUTED_DATASET_DIR, gene)
        gene_windows[gene] = {"chrom": chrom, "start": start, "end": end}
else:
    pd_setup.ensure_fasta_index(pd_setup.DEFAULT_REF_FASTA)
    chr_prefix = pd_setup.detect_chr_prefix(pd_setup.DEFAULT_REF_FASTA)
    print(f"Chromosome prefix: {chr_prefix!r}")

    print(f"Loading GTF from: {pd_setup.GTF_URL}")
    gtf = pd.read_feather(pd_setup.GTF_URL)

    for gene in pd_setup.GENES:
        chrom, start, end = pd_setup.gene_window(gtf, gene, chr_prefix)
        gene_windows[gene] = {"chrom": chrom, "start": start, "end": end}

pd.DataFrame(gene_windows).T

## Step 3: reference windows + per-gene variant slices

For each gene: get its reference window FASTA (copied directly from `PRECOMPUTED_DATASET_DIR` if
set, otherwise extracted from the FASTA with `samtools faidx`), then slice its window from the
matching chromosome VCF with `bcftools view`, restricted to the individuals from Step 1 and with
unsupported symbolic ALTs (other than `<DEL>`) filtered out -- the same filter
`build_window_and_predict.py` applies before `bcftools consensus`. Both steps are skipped if their
output already exists, so this cell is safe to re-run.

In [ ]:
pd_setup.DATA_DIR.mkdir(parents=True, exist_ok=True)
target_populations = {pop for pops in pd_setup.CLASS_MAP.values() for pop in pops}
sample_ids = sorted(metadata_df.loc[metadata_df["Population"].isin(target_populations), "SampleID"])
print(f"{len(sample_ids)} individuals will be included in each gene's variants VCF")

for gene, window in gene_windows.items():
    print(f"\n[{gene}] window = {window['chrom']}:{window['start']}-{window['end']}")
    ref_out = pd_setup.DATA_DIR / "references" / gene / "ref.window.fa"
    if PRECOMPUTED_DATASET_DIR is not None:
        pd_setup.copy_ref_window_fasta(PRECOMPUTED_DATASET_DIR, gene, ref_out)
    else:
        pd_setup.extract_ref_window_fasta(
            pd_setup.DEFAULT_REF_FASTA, window["chrom"], window["start"], window["end"], ref_out,
        )
    # kept as a plain str, not wrapped in Path() -- Path() would mangle a "http://" URL's
    # double slash into "http:/", silently breaking remote VCF access.
    vcf_path = pd_setup.DEFAULT_VCF_PATTERN.format(chrom=window["chrom"])
    pd_setup.slice_gene_variants(gene, window["chrom"], window["start"], window["end"], vcf_path, sample_ids)

## Step 4: write `notebooks/data/` metadata files

In [ ]:
pd_setup.write_experiment_json()
individuals = pd_setup.write_individuals_json(metadata_df)
pd_setup.write_genes_json(gene_windows)

sex_counts = pd.Series([info["sex"] for info in individuals.values()]).value_counts()
family_count = len({info["family_id"] for info in individuals.values()})
print(f"\n{len(individuals)} individuals written, {family_count} distinct family IDs, sex breakdown: {sex_counts.to_dict()}")
print("\n[DONE] notebooks/data is ready -- simplified_notebook.ipynb can now run without touching\n"
      "the external dataset mount, reference FASTA, or chromosome VCFs.")

## Step 5 (optional): import precomputed predictions -- avoids AlphaGenome API calls

Only applicable if `PRECOMPUTED_DATASET_DIR` was found in Step 2 (this machine's own pre-built
dataset) -- skipped automatically otherwise (e.g. on a fresh clone with no local genomics data;
`simplified_notebook.ipynb`'s own `generate_predictions()` will call the AlphaGenome API instead).

When applicable: that dataset already has, for every individual, AlphaGenome RNA-seq predictions on
their own H1/H2 consensus sequence for each gene -- computed once when it was built. Rather than
re-predicting the same sequences through the API (23,600+ calls at full scope: 11 genes x 1,072
individuals x 2 haplotypes), this step copies those raw predictions and realigns them onto
reference coordinates with this repo's *current* indel/`<DEL>`-aware logic
(`notebooks/utils/realignment.py`), writing to `notebooks/.cache/predictions/` -- the exact cache
`simplified_notebook.ipynb`'s `generate_predictions()` reads from and writes to. Once imported, that
notebook's demo-generation cell will report every imported (gene, sample, haplotype) as `"cached"`
and skip it, with no API calls at all.

Local-only (no network calls) but reads/writes a lot of data -- full scope is on the order of tens
of GB in `notebooks/.cache/predictions/` (gitignored, not committed). Resumable like every other
step here: safe to interrupt and re-run, or to widen `IMPORT_GENES`/`IMPORT_SAMPLE_IDS`
incrementally.

In [ ]:
if PRECOMPUTED_DATASET_DIR is None:
    print("PRECOMPUTED_DATASET_DIR not set (see Step 2) -- nothing to import on this machine. "
          "simplified_notebook.ipynb's generate_predictions() will use the AlphaGenome API instead.")
    import_results = []
else:
    from utils import import_precomputed

    # Full scope by default (all genes, all individuals) -- this is local-only (no API cost), so
    # unlike simplified_notebook.ipynb's own demo cell there's no reason to start small. Narrow
    # IMPORT_GENES / IMPORT_SAMPLE_IDS if you only need a subset.
    IMPORT_GENES = pd_setup.GENES
    IMPORT_SAMPLE_IDS = sample_ids

    PREDICTIONS_CACHE_DIR = NOTEBOOK_DIR / ".cache" / "predictions"

    def _on_progress(done, total, gene, sample_id, haplotype, status):
        if done % 200 == 0 or done == total:
            print(f"[{done}/{total}] last: {gene} {sample_id} {haplotype}: {status}")

    import_results = import_precomputed.import_precomputed_predictions(
        PRECOMPUTED_DATASET_DIR,
        genes=IMPORT_GENES,
        gene_rows=gene_windows,
        sample_ids=IMPORT_SAMPLE_IDS,
        output_dir=PREDICTIONS_CACHE_DIR,
        on_progress=_on_progress,
    )

    status_counts = pd.Series([r["status"] for r in import_results]).value_counts()
    print(f"\n{status_counts.to_dict()}")